# Compare Output Segmentations

Let's compare U-Net outputs with NVAE-Corrector outputs.

In [ ]:
import os

# Change this variable if the root folder name has been changed
root_dir = "nvae-shape-encoding"
current_dir = os.getcwd()

if not current_dir.endswith(root_dir):
    %cd ..

assert os.getcwd().endswith(root_dir)

We don't have to perform inference for U-Net, since the predictions are already
saved for the Mask-to-Mask pipeline.

Update `model_path` to choose the model.

In [ ]:
from arch.nvae_corrector.nvae_corrector import NVAECorrector
from arch.vae_corrector.vae_corrector import VAECorrector
from arch.cnvae.cnvae import CNVAE
from arch.unet.acunet import ACUNet
from arch.unet.acunet_vae import ACVAEUNet

nvae_m2m_extended = "logs/nvae_corrector_acdc/default-extended/checkpoints/epoch=20-step=17976.ckpt"
# vae_m2m = "logs/vae_corrector_acdc/best/checkpoints/epoch=40-step=4387.ckpt"
vae_m2m = "logs/vae_corrector_acdc/extended/b-2-latent-32/checkpoints/epoch=11-step=5136.ckpt"
nvae_acunet = "logs/unet_acdc/default-shape-prior/checkpoints/epoch=67-step=3672.ckpt"
vae_acunet = "logs/unet_acdc/shape-prior-vae-baselines/shape-prior-vae-seed-1974/checkpoints/epoch=64-step=3510.ckpt"
cnvae_i2m = "logs/cnvae_acdc/best/checkpoints/epoch=83-step=17976.ckpt"


In [ ]:
import lightning as L
import torch

from utils.const import SEED
from data_modules.acdc import ACDC3DWithPredictedMaskDataModule
from utils.utils import setup_device

# Setup device
device = setup_device()
print(f"Device: {device}")

# Seed
L.seed_everything(SEED)

# Load data
data_module = ACDC3DWithPredictedMaskDataModule()

# Load models
models_nvae_m2m_extended = NVAECorrector.load_from_checkpoint(nvae_m2m_extended)
models_vae_m2m = VAECorrector.load_from_checkpoint(vae_m2m)
models_nvae_acunet = ACUNet.load_from_checkpoint(nvae_acunet)
models_vae_acunet = ACVAEUNet.load_from_checkpoint(vae_acunet)
models_cnvae_i2m = CNVAE.load_from_checkpoint(cnvae_i2m)

# Reseed after preprocessing data
L.seed_everything(SEED)

In [ ]:
loader_test = data_module.test_dataloader()
y, y_pred, _, _ = next(iter(loader_test))
print(y.shape, y_pred.shape)

## View Segmentations

Let's seek the worst U-Net predictions in terms of anatomical validity, and see
how NVAE-Corrector improves them.

In [ ]:
# Do not use a data loader, since scans are encapsulated. Furthermore, not
# further preprocessing is done in Dataset.__getitem__.

data = data_module.data_test

print(data.scans[0].shape)
print(data.masks[0].shape)
print(data.masks_pred[0].shape)

print(len(data))

In [ ]:
from utils.anatomical_validity_checker import AnatomicalValidityChecker
from utils.eval import compute_dice_score
from utils.utils import discretise

loader_test = data_module.test_dataloader()

worst_x = []
worst_y = []
worst_y_pred = []
worst_y_hat_nvae_m2m_extended = []
worst_y_hat_vae_m2m = []
worst_y_hat_nvae_acunet = []
worst_y_hat_vae_acunet = []
worst_y_hat_cnvae_i2m = []

with torch.no_grad():
    # models_nvae_acunet = ACUNet.load_from_checkpoint(nvae_acunet)
    # models_vae_acunet = ACVAEUNet.load_from_checkpoint(vae_acunet)
    # models_cnvae_i2m = CNVAE.load_from_checkpoint(cnvae_i2m)
    
    models = [models_nvae_m2m_extended, models_vae_m2m, models_nvae_acunet, models_vae_acunet, models_cnvae_i2m]
    
    for model in models:
        model.eval()
        model.to(device)
    
    for i in range(len(data)):
    # for i in range(10):
        # 3D data
        xs = data.scans[i]
        ys = data.masks[i]
        y_preds = data.masks_pred[i]
        
        for x, y, y_pred in zip(xs, ys, y_preds):
            # Individual 2D slice
            x = x.unsqueeze(0)
            y = y.unsqueeze(0)
            y_pred = y_pred.unsqueeze(0)

            x = x.to(device)
            y = y.to(device)
            y_pred = y_pred.to(device)

            y_hat_logits_nvae_m2m_extended, _, _, _, _ = models_nvae_m2m_extended(y_pred, test=True)
            y_hat_onehot_nvae_m2m_extended = discretise(y_hat_logits_nvae_m2m_extended)
            
            _, _, _, y_hat_logits_vae_m2m = models_vae_m2m(y_pred, test=True)
            y_hat_onehot_vae_m2m = discretise(y_hat_logits_vae_m2m)
            
            y_hat_logits_nvae_acunet = models_nvae_acunet(x)
            y_hat_onehot_nvae_acunet = discretise(y_hat_logits_nvae_acunet)
            
            y_hat_logits_vae_acunet = models_vae_acunet(x)
            y_hat_onehot_vae_acunet = discretise(y_hat_logits_vae_acunet)
            
            y_hat_logits_cnvae_i2m = models_cnvae_i2m.inference(x)
            y_hat_onehot_cnvae_i2m = discretise(y_hat_logits_cnvae_i2m)
            
            # Condition: Anatomically invalid U-Net prediction
            
            for discretised_y_hat in y_pred:
                AV = AnatomicalValidityChecker(discretised_y_hat)

                if AV.count_violations() > 1:
                    worst_x.append(x)
                    worst_y.append(y)
                    worst_y_pred.append(y_pred)
                    worst_y_hat_nvae_m2m_extended.append(y_hat_onehot_nvae_m2m_extended)
                    worst_y_hat_vae_m2m.append(y_hat_onehot_vae_m2m)
                    worst_y_hat_nvae_acunet.append(y_hat_onehot_nvae_acunet)
                    worst_y_hat_vae_acunet.append(y_hat_onehot_vae_acunet)
                    worst_y_hat_cnvae_i2m.append(y_hat_onehot_cnvae_i2m)

print(len(worst_x))

In [ ]:
# Choose the samples to view

# samples_idx = torch.arange(0, len(worst_x))

# Figure 1
#                             0  1  2   3   4   5   6   7   8   9       11      13      15      17
# samples_idx = torch.Tensor([2, 9, 10, 11, 18, 19, 21, 33, 34, 35, 39, 48, 49, 50, 63, 66, 71, 74]).int()
samples_idx = torch.Tensor([11, 66, 21, 71]).int()

# Figure 2
#                                1       3       5       7
# samples_idx = torch.Tensor([7, 51, 55, 68, 69, 73, 48, 34, 39]).int()
# samples_idx = torch.Tensor([48, 34]).int()
samples_idx = torch.Tensor([51]).int()
# samples_idx = torch.Tensor([69]).int()


worst_x_subset = torch.stack([worst_x[i] for i in samples_idx]).squeeze(1)
worst_y_subset = torch.stack([worst_y[i] for i in samples_idx]).squeeze(1)
worst_y_pred_subset = torch.stack([worst_y_pred[i] for i in samples_idx]).squeeze(1)
worst_y_hat_subset_nvae_m2m_extended = torch.stack([worst_y_hat_nvae_m2m_extended[i] for i in samples_idx]).squeeze(1)
worst_y_hat_subset_vae_m2m = torch.stack([worst_y_hat_vae_m2m[i] for i in samples_idx]).squeeze(1)
worst_y_hat_subset_nvae_acunet = torch.stack([worst_y_hat_nvae_acunet[i] for i in samples_idx]).squeeze(1)
worst_y_hat_subset_vae_acunet = torch.stack([worst_y_hat_vae_acunet[i] for i in samples_idx]).squeeze(1)
worst_y_hat_subset_cnvae_i2m = torch.stack([worst_y_hat_cnvae_i2m[i] for i in samples_idx]).squeeze(1)

(
    worst_x_subset.shape,
    worst_y_subset.shape,
    worst_y_pred_subset.shape,
    worst_y_hat_subset_nvae_m2m_extended.shape,
    worst_y_hat_subset_vae_m2m.shape,
    worst_y_hat_subset_nvae_acunet.shape,
    worst_y_hat_subset_vae_acunet.shape,
    worst_y_hat_subset_cnvae_i2m.shape,
)

In [ ]:
corrector_ids = [
    r"$\text{U-Net}$",
    r"$\text{CVAE}_{\text{Edit}}$",
    r"$\text{CNVAE}_{\text{Edit}}$",
]

worst_y_hat_subsets = [
    worst_y_pred_subset,
    worst_y_hat_subset_vae_m2m,
    worst_y_hat_subset_nvae_m2m_extended,
]

In [ ]:
# Compute Dice score and AV

corrector_dscs = []
corrector_violations = []

for i, y_hat_onehot in enumerate(worst_y_hat_subsets):
    dscs_arr = []
    violations_arr = []
    
    # Go through each slice
    for j, y_hat_onehot_slice in enumerate(y_hat_onehot):
        y_hat_onehot_slice = y_hat_onehot_slice.unsqueeze(0)
        y = worst_y_subset[j].unsqueeze(0)
        
        dice_score, dice_score_per_class = compute_dice_score(
            y,
            y_hat_onehot_slice,
            device,
            is_3d=True,
            dice_per_class=True,
        )
        
        # Compute anatomical violations
        
        AV = AnatomicalValidityChecker(y_hat_onehot_slice.squeeze(0))
        violations = AV.count_violations()
        
        # Save the results
        
        dscs_arr.append(dice_score)
        violations_arr.append(violations)
    
    dscs_arr = torch.Tensor(dscs_arr)
    violations_arr = torch.Tensor(violations_arr)

    corrector_dscs.append(dscs_arr)
    corrector_violations.append(violations_arr)
        

In [ ]:
from utils.eval import get_samples_and_reconstructions_pixel_diff


ground_truth, predictions, reconstruction_pixel_error = get_samples_and_reconstructions_pixel_diff(
    worst_y_subset,
    worst_y_pred_subset,
    return_reconstructions=True,
)

segmentation_corrections = dict()

for corrector_id, worst_y_hat_subset, dscs, violations in zip(
    corrector_ids,
    worst_y_hat_subsets,
    corrector_dscs,
    corrector_violations,
):
    _, corrections, _ = get_samples_and_reconstructions_pixel_diff(
        worst_y_subset,
        worst_y_hat_subset,
        return_reconstructions=True,
    )
    
    segmentation_corrections[corrector_id] = (corrections, dscs, violations)

In [ ]:
for i, (corrector_id, my_info) in enumerate(segmentation_corrections.items()):
    corrections, dscs, violations = my_info
    
    print(corrector_id, corrections.shape, dscs.shape, violations.shape)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from torchvision.utils import make_grid

from utils.colourmap import GRGB

num_samples = len(samples_idx)

fig, ax = plt.subplots(1, 4, figsize=(24, 6 * num_samples))

# ==============================================================================
# EDIT HERE
# ==============================================================================
    
factor = 1 / num_samples
x_offset = 0.028
y_offset = 0.984
y_offset_2 = 0.868
# word_alpha = 0.64
word_alpha = 0.4
label_size = 30
toplevel_font = 48

scans = worst_x_subset.cpu().float()
scans = make_grid(scans, nrow=1, padding=2)
scans = np.transpose(scans.numpy(), (1, 2, 0))

# Column 1: Ground truth

ground_truth = ground_truth.cpu().float()
ground_truth_grid = make_grid(ground_truth, nrow=1, padding=2, normalize=True)
ground_truth_grid = ground_truth_grid[0]

ax[0].axis("off")
ax[0].imshow(ground_truth_grid, cmap=GRGB)
ax[0].set_title("Ground Truth", fontsize=toplevel_font, pad=18)

# for row in range(num_samples):
#     text = f"INDEX {row}"
    
#     ax[0].text(
#         x_offset, y_offset - row * factor, text, transform=ax[0].transAxes,
#         fontsize=label_size, verticalalignment='top', color='white', 
#         bbox=dict(facecolor='black', alpha=word_alpha, edgecolor='none')
#     )

# Column 2 to 4: U-Net + Corrections

for i, (corrector_id, my_info) in enumerate(segmentation_corrections.items()):
    corrections, dscs, violations = my_info
    
    dscs = dscs.cpu().numpy()
    violations = violations.cpu().numpy()
    
    corrections = corrections.cpu().float()
    corrections_grid = make_grid(corrections, nrow=1, padding=2, normalize=True)
    corrections_grid = corrections_grid[0]

    ax[i + 1].axis("off")
    ax[i + 1].imshow(corrections_grid, cmap=GRGB)
    ax[i + 1].set_title(corrector_id, fontsize=toplevel_font, pad=18)
    
    # Show text on top left of each image
    
    for row in range(len(dscs)):
        dsc = dscs[row]
        violation = violations[row]

        dsc_text = f"DSC: {dsc * 100:.1f}"
        violations_text = f"ERR: {violation:.0f}"
        
        ax[i + 1].text(
            x_offset, y_offset - row * factor, dsc_text, transform=ax[i + 1].transAxes,
            fontsize=label_size, verticalalignment='top', color='white', 
            bbox=dict(facecolor='black', alpha=word_alpha, edgecolor='none')
        )
        
        ax[i + 1].text(
            x_offset, y_offset_2 - row * factor, violations_text, transform=ax[i + 1].transAxes,
            fontsize=label_size, verticalalignment='top', color='white', 
            bbox=dict(facecolor='black', alpha=word_alpha, edgecolor='none')
        )

plt.subplots_adjust(wspace=0, hspace=0)